# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('/content/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [3]:
# Task 1
# -------
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

ratings_keep = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
sub = df[df['rating'].isin(ratings_keep)]

pivot = (sub.groupby(['rating', 'decade']).size()
         .unstack(fill_value=0)
         .reindex(ratings_keep))  # keep a consistent, meaningful row order

fig1 = px.imshow(
    pivot,
    color_continuous_scale='Blues',
    text_auto=True,
    aspect='auto',
    labels=dict(x='Release decade', y='Content rating', color='Number of titles'),
)

peak_rating = pivot.max(axis=1).idxmax()
peak_decade = pivot.loc[peak_rating].idxmax()
peak_value = pivot.loc[peak_rating, peak_decade]

fig1.update_layout(
    title=dict(
        text=(f"<b>{peak_rating} dominates every decade, peaking at {peak_value} titles in the {peak_decade}</b>"
              f"<br><sup>Number of titles by content rating and release decade</sup>"),
        x=0, xanchor='left', font=dict(size=16)
    ),
    font=dict(family='Arial, sans-serif', size=13, color='#333'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=450,
    width=750,
    margin=dict(l=10, r=10, t=90, b=40),
)
fig1.update_xaxes(showgrid=False)
fig1.update_yaxes(showgrid=False)

fig1.show()


## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [4]:
# Task 2
# -------
movies = df[df['type'] == 'Movie']

yearly = (movies.groupby('added_year').size()
          .reindex(range(2015, 2023), fill_value=0))

years = [str(y) for y in yearly.index] + ['Total']
values = list(yearly.values) + [yearly.sum()]
measures = ['relative'] * len(yearly) + ['total']

fig2 = go.Figure(go.Waterfall(
    x=years,
    y=values,
    measure=measures,
    increasing=dict(marker=dict(color='#2E7D32')),   # green for additions
    totals=dict(marker=dict(color='#2E86AB')),        # blue for the total
    text=[str(v) for v in values],
    textposition='outside',
    connector=dict(line=dict(color='#cccccc', width=1)),
))

peak_year = yearly.idxmax()
peak_val = yearly.max()

fig2.add_annotation(
    x=str(peak_year), y=peak_val,
    text=f'<b>{peak_year}: largest single-year addition ({peak_val} movies)</b>',
    showarrow=True, arrowhead=2, ax=0, ay=-40,
    font=dict(size=12, color='#2E7D32'),
)

fig2.update_layout(
    title=dict(
        text=(f"<b>Netflix added {yearly.sum()} movies between 2015 and 2022, led by a {peak_year} surge</b>"
              f"<br><sup>Movies added to the catalogue, by year, with running total</sup>"),
        x=0, xanchor='left', font=dict(size=16)
    ),
    xaxis=dict(title=''),
    yaxis=dict(title='Movies added', showgrid=True, gridcolor='#eeeeee', zeroline=False),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=13, color='#333'),
    height=500,
    width=850,
    margin=dict(l=10, r=10, t=90, b=40),
    showlegend=False,
)

fig2.show()
